# Day 7: Hardening the benchmark - when a perfect Score is a Red Flag

**Deep Coding: Real Hard Negatives + Paraphrased Questions**
**Hard Negatives, Not Random Distractors:** Instead of oversampling the same section queries, pull distractor chunks that are topically close to each benchmark chunk but factually different, same section, different fiscal year or different line item, genuinely confusable, not just **"*something else from the document*"**
**Paraphrase the Questions:** Reword each nemchmark question so it nolonger echoes it's source chunk's exact phrasing, testing semantic matching instead of keyword overlap.
**Return HitRate@4:** on the hardened set, across the same three models, and see if the real gap finally shows up.


In [2]:
# setting up vector db connection before i set up the vector engine 
from groq import Groq
from dataclasses import dataclass, field, asdict
from sentence_transformers import SentenceTransformer
import numpy as np
from dotenv import load_dotenv, find_dotenv
import os
from llama_index.vector_stores.mongodb import MongoDBAtlasVectorSearch
from llama_index.storage.docstore.mongodb import MongoDocumentStore
from pymongo import MongoClient, AsyncMongoClient
from llama_index.core import VectorStoreIndex, StorageContext
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import Settings

Settings.embed_model = HuggingFaceEmbedding(
    model_name= 'BAAI/bge-m3'
)

load_dotenv(find_dotenv())
client = Groq()

# Re-establishing the connection to my mongoDB atlas vector store to retrieve historical data

# mongoDB atlas connection
mongoClient = MongoClient(os.getenv('MONGO_URI'))
asyncMongoClient = AsyncMongoClient(os.getenv('MONGO_URI'))


# reconnecting persistent docstore
docstore = MongoDocumentStore.from_uri(
    uri = os.getenv('MONGO_URI'),
    db_name = 'month1_database',
    namespace = 'month_1_collection'
)     


# 1. Connect to your MongoDB
vectorStore = MongoDBAtlasVectorSearch(
    mongodb_client=mongoClient,
    async_mongodb_client= asyncMongoClient, 
    db_name= 'month1_database',
    collection_name= 'month_1_rag_collection_v3',
    vector_index_name= 'final_index_v3',
    embedding_key= 'embedding'
)

storageContext = StorageContext.from_defaults(
    vector_store= vectorStore,
    docstore= docstore
)

index = VectorStoreIndex.from_vector_store(
    vector_store= vectorStore,
    storage_context = storageContext
)



print('🤖🛩️ Vector Store connection established ⚡')

c:\Users\rodne\miniconda3\envs\rag-Ai\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🤖🛩️ Vector Store connection established ⚡


In [3]:
RELEVANCE_THRESHOLD = 0.5

def retrieve_with_confidence(query: str) -> tuple[list, float]:
    """Returns retrieved docs and the top chunk's confidence score.
    Uses cosine similarity to score from your vectore store.
    """

    # creating a native llamaindex retriever from my initialized index
    retriever = index.as_retriever(similarity_top_k = 4)
    results = retriever.retrieve(query)

    # reults is a list of (document, score) tuples
    # lower score = more similar in FAISS  (L2 distance); invert if needed
    # For Cosine similarity stores, higher = better

    if not results:
        return [], 0.0
    
    
    top_score = float(results[0].score) if results[0].score is not None else 0.0

    print(f"📣 Top retrieval score : {top_score:.3f} (threshold: {RELEVANCE_THRESHOLD})")
    return results, top_score

In [4]:
# PULLING CHUNKS SPREAD ACROSS THE DOCUMENT
# TOKEN USER TRACKER

@dataclass
class TokenUsage:
    prompt_tokens: int = 0
    completion_tokens: int = 0
    total_calls: int = 0

    def add(self, usage):
        self.prompt_tokens += usage.prompt_tokens
        self.completion_tokens += usage.completion_tokens
        self.total_calls += 1
    
    def cost_estimate(
            self,
            input_price_per_1m: float = 2.50,
            output_price_per_1m: float = 10.00
    ) -> float:
        input_cost = (self.prompt_tokens /1000000) * input_price_per_1m
        output_cost = (self.completion_tokens /1000000) * output_price_per_1m
        return round(input_cost + output_cost, 6)
    
    def report(self):
        print(f"\n📣 Token usage report")
        print(f"LLM calls : {self.total_calls}")
        print(f"Prompt tokens: {self.prompt_tokens}")
        print(f"Completion tokens: {self.completion_tokens}")
        print(f"Total tokens: {self.prompt_tokens + self.completion_tokens:,}")
        print(f"Est. cost (GPT-40 pricing): ${self.cost_estimate()}")


# global tracher that is set before each run
usage_tracker = TokenUsage()


SECTION_QUERIES = [
    "business overview and products",
    "risk factors",
    "management discussion and analysis",
    "consolidated statements of operations",
    "consolidated balance sheet",
    "notes to financial statements",
]

def sample_chunks_across_sections(chunks_per_section: int = 3) -> list[dict]:
    sampled = []
    for section_query in SECTION_QUERIES:
        docs, score = retrieve_with_confidence(section_query)
        for doc in docs[:chunks_per_section]:
            sampled.append({
                "section": section_query,
                "content": doc.node.get_content() if hasattr(doc, "node") else doc.page_content,
                "retrieval_score": score,
            })
    print(f"📋 Sampled {len(sampled)} chunks across {len(SECTION_QUERIES)} sections")
    return sampled

sampled_chunks = sample_chunks_across_sections(chunks_per_section=3)

def tracked_llm_call(messages: list, system: str = "") -> str:
    """
    Wraps every Groq call so token usage is always captured.
    Drop-in replacement for direct client.chat.completions.create calls.
    
    """

    full_messages = []
    if system:
        full_messages.append({"role": "system", "content": system})
    full_messages.extend(messages)

    response = client.chat.completions.create(
        model = "openai/gpt-oss-120b",
        messages= full_messages,
        temperature= 0,
    )

    usage_tracker.add(response.usage)
    return response.choices[0].message.content

📣 Top retrieval score : 0.798 (threshold: 0.5)
📣 Top retrieval score : 0.813 (threshold: 0.5)
📣 Top retrieval score : 0.775 (threshold: 0.5)
📣 Top retrieval score : 0.828 (threshold: 0.5)
📣 Top retrieval score : 0.829 (threshold: 0.5)
📣 Top retrieval score : 0.797 (threshold: 0.5)
📋 Sampled 18 chunks across 6 sections


In [5]:
# Fixing Day 3's gap, Generating aSmall, Chunk Traceable Benchmark Set

import re
import json
from typing import Optional

CHUNK_TRACEABLE_GENERATOR_PROMPT = """You are generating an evaluation question for a
RAG system testing an Apple FY2024 10-K.

You will receive ONE chunk of real text. Write exactly one question answerable COMPLETELY
from this chunk alone, plus the correct answer.

Respond in EXACTLY this format:
QUESTION: <the question, or SKIP>
ANSWER: <the ground truth answer, or SKIP>
"""

def generate_traceable_pair(chunk: dict) -> Optional[dict]:
    raw = tracked_llm_call(
        messages=[{"role": "user", "content": f"Chunk:\n{chunk['content']}"}],
        system=CHUNK_TRACEABLE_GENERATOR_PROMPT
    )
    q_match = re.search(r"QUESTION:\s*(.+)", raw)
    a_match = re.search(r"ANSWER:\s*(.+)", raw)
    if not q_match or not a_match:
        return None
    question, answer = q_match.group(1).strip(), a_match.group(1).strip()
    if question.upper() == "SKIP":
        return None
    # THE FIX: save the exact source chunk text, not just its section label
    return {
        "question": question,
        "ground_truth": answer,
        "source_chunk_content": chunk["content"],
        "source_section": chunk["section"],
    }

benchmark_chunks = sample_chunks_across_sections(chunks_per_section=3)
traceable_pairs = []
for chunk in benchmark_chunks:
    pair = generate_traceable_pair(chunk)
    if pair:
        traceable_pairs.append(pair)
        print(f"✅ [{pair['source_section']}] {pair['question'][:65]}")

print(f"\n📊 Built {len(traceable_pairs)} chunk-traceable benchmark questions")

📣 Top retrieval score : 0.798 (threshold: 0.5)
📣 Top retrieval score : 0.813 (threshold: 0.5)
📣 Top retrieval score : 0.775 (threshold: 0.5)
📣 Top retrieval score : 0.828 (threshold: 0.5)
📣 Top retrieval score : 0.829 (threshold: 0.5)
📣 Top retrieval score : 0.797 (threshold: 0.5)
📋 Sampled 18 chunks across 6 sections
✅ [business overview and products] What types of products does the Company design, manufacture, and 
✅ [business overview and products] To which types of customers does the Company sell its products an
✅ [business overview and products] What market conditions are described for the Company’s products a
✅ [risk factors] Which sections of the Form 10‑K does the risk factors discussion 
✅ [risk factors] According to the Risk Factors section, which aspects of the Compa
✅ [risk factors] What does the discussion of risk factors contain?
✅ [management discussion and analysis] According to the passage, what makes auditing management’s evalua
✅ [management discussion and analysis] 

In [6]:
# Generating hard negatives, same section, different Fact

def get_hard_negative(true_chunk: dict, section_query: str) -> Optional[str]:
    """Pulls a chunk from the SAME section but checks it's not the same fact."""
    docs, _ = retrieve_with_confidence(section_query)
    candidates = [d.node.get_content() for d in docs if d.node.get_content() != true_chunk["content"]]
    return candidates[0] if candidates else None

benchmark_chunks = sample_chunks_across_sections(chunks_per_section=3)
hardened_pairs = []

for chunk in benchmark_chunks:
    pair = generate_traceable_pair(chunk)  # reused from Day 6, unchanged
    if not pair:
        continue
    hard_negative = get_hard_negative(chunk, chunk["section"])
    if hard_negative:
        pair["hard_negative"] = hard_negative
        hardened_pairs.append(pair)

print(f"📊 Built {len(hardened_pairs)} question/chunk pairs, each with a real same-section hard negative")

📣 Top retrieval score : 0.798 (threshold: 0.5)
📣 Top retrieval score : 0.813 (threshold: 0.5)
📣 Top retrieval score : 0.775 (threshold: 0.5)
📣 Top retrieval score : 0.828 (threshold: 0.5)
📣 Top retrieval score : 0.829 (threshold: 0.5)
📣 Top retrieval score : 0.797 (threshold: 0.5)
📋 Sampled 18 chunks across 6 sections
📣 Top retrieval score : 0.798 (threshold: 0.5)
📣 Top retrieval score : 0.798 (threshold: 0.5)
📣 Top retrieval score : 0.798 (threshold: 0.5)
📣 Top retrieval score : 0.813 (threshold: 0.5)
📣 Top retrieval score : 0.813 (threshold: 0.5)
📣 Top retrieval score : 0.813 (threshold: 0.5)
📣 Top retrieval score : 0.775 (threshold: 0.5)
📣 Top retrieval score : 0.775 (threshold: 0.5)
📣 Top retrieval score : 0.775 (threshold: 0.5)
📣 Top retrieval score : 0.828 (threshold: 0.5)
📣 Top retrieval score : 0.828 (threshold: 0.5)
📣 Top retrieval score : 0.828 (threshold: 0.5)
📣 Top retrieval score : 0.829 (threshold: 0.5)
📣 Top retrieval score : 0.829 (threshold: 0.5)
📣 Top retrieval score 

In [7]:
# PARAPHRASING EACH question away from it's source wording.

PARAPHRASE_PROMPT = """Reword this question so it asks the same thing but uses DIFFERENT
words and phrasing than a typical 10-K document would use. Keep the meaning identical.

Respond with ONLY the reworded question, nothing else.
"""

for pair in hardened_pairs:
    paraphrased = tracked_llm_call(
        messages=[{"role": "user", "content": pair["question"]}],
        system=PARAPHRASE_PROMPT
    )
    pair["original_question"] = pair["question"]
    pair["question"] = paraphrased.strip()

print("✅ All questions paraphrased away from source-chunk wording")
print(f"   Example: '{hardened_pairs[0]['original_question']}'")
print(f"        →   '{hardened_pairs[0]['question']}'")

✅ All questions paraphrased away from source-chunk wording
   Example: 'What types of products does the Company design, manufacture, and market?'
        →   'Which categories of goods does the firm develop, produce, and sell?'


In [8]:
# Building a pool where hard negatives actually compete

true_chunks = {p["source_chunk_content"] for p in hardened_pairs}
hard_negative_chunks = {p["hard_negative"] for p in hardened_pairs}
chunk_pool = list(true_chunks | hard_negative_chunks)

print(f"📋 Hardened pool: {len(chunk_pool)} chunks "
      f"({len(true_chunks)} true answers + {len(hard_negative_chunks)} genuine hard negatives)")

📋 Hardened pool: 17 chunks (17 true answers + 11 genuine hard negatives)


In [9]:

# Hit Rate@K, Computed Identically for Any Model

def compute_hit_rate(model: SentenceTransformer, model_name: str, k: int = 4) -> dict:
    print(f"\n{'='*60}\nEvaluating: {model_name}\n{'='*60}")

    pool_embeddings = model.encode(chunk_pool, show_progress_bar=True, normalize_embeddings=True)

    hits = 0
    per_question_results = []

    for pair in traceable_pairs:
        query_embedding = model.encode([pair["question"]], normalize_embeddings=True)[0]
        similarities = pool_embeddings @ query_embedding  # cosine similarity, since normalized
        top_k_indices = np.argsort(similarities)[::-1][:k]
        top_k_chunks = [chunk_pool[i] for i in top_k_indices]

        is_hit = pair["source_chunk_content"] in top_k_chunks
        hits += int(is_hit)
        per_question_results.append({"question": pair["question"], "hit": is_hit})
        print(f"  {'✅' if is_hit else '❌'} {pair['question'][:60]}")

    hit_rate = hits / len(traceable_pairs)
    print(f"\n📊 {model_name} — Hit Rate@{k}: {hits}/{len(traceable_pairs)} ({hit_rate*100:.1f}%)")
    return {"model_name": model_name, "hit_rate": hit_rate, "results": per_question_results}

In [10]:
# Rerunning HIT Rate@4 on the hardened set

CANDIDATE_MODELS = {
    "BAAI/bge-m3": "Current production model (multilingual, long-context)",
    "sentence-transformers/all-MiniLM-L6-v2": "Small, fast, general-purpose baseline",
    "BAAI/bge-large-en-v1.5": "Strong English-only alternative",
}

traceable_pairs = hardened_pairs  # compute_hit_rate reads this name, reused unchanged from Day 6

hardened_model_results = []
for model_id, description in CANDIDATE_MODELS.items():
    print(f"\n📥 Loading {model_id}")
    model = SentenceTransformer(model_id)
    result = compute_hit_rate(model, model_id, k=4)
    hardened_model_results.append(result)
    del model


📥 Loading BAAI/bge-m3

Evaluating: BAAI/bge-m3


Batches: 100%|██████████| 1/1 [00:54<00:00, 54.58s/it]


  ✅ Which categories of goods does the firm develop, produce, an
  ✅ Which categories of customers does the Company sell its prod
  ✅ What primary traits define the worldwide markets where the c
  ✅ In the risk factors section, how does the Company explain th
  ✅ Based on the Risk Factors portion, which elements of the Com
  ✅ What kind of statements are presented in the risk factors di
  ✅ Based on the excerpt, why is it challenging to audit managem
  ✅ Which particular procedures did the firm evaluate concerning
  ✅ Which elements might this evaluation be built upon?
  ✅ Which page of Apple’s FY 2024 Form 10‑K includes the Consoli
  ✅ Based on Note 1, whose financial records are incorporated in
  ✅ In the Index to the combined financial statements, which pag
  ✅ Which page of Apple’s FY 2024 Form 10‑K contains the Consoli
  ✅ In 2024, what was the aggregate amount of other current liab
  ✅ What amount of total equity did shareholders have on Septemb
  ✅ When are taxes allowed to be s

Batches: 100%|██████████| 1/1 [00:01<00:00,  1.94s/it]


  ✅ Which categories of goods does the firm develop, produce, an
  ✅ Which categories of customers does the Company sell its prod
  ✅ What primary traits define the worldwide markets where the c
  ✅ In the risk factors section, how does the Company explain th
  ✅ Based on the Risk Factors portion, which elements of the Com
  ✅ What kind of statements are presented in the risk factors di
  ✅ Based on the excerpt, why is it challenging to audit managem
  ✅ Which particular procedures did the firm evaluate concerning
  ✅ Which elements might this evaluation be built upon?
  ✅ Which page of Apple’s FY 2024 Form 10‑K includes the Consoli
  ✅ Based on Note 1, whose financial records are incorporated in
  ✅ In the Index to the combined financial statements, which pag
  ✅ Which page of Apple’s FY 2024 Form 10‑K contains the Consoli
  ✅ In 2024, what was the aggregate amount of other current liab
  ✅ What amount of total equity did shareholders have on Septemb
  ✅ When are taxes allowed to be s

Batches: 100%|██████████| 1/1 [00:37<00:00, 37.85s/it]


  ✅ Which categories of goods does the firm develop, produce, an
  ✅ Which categories of customers does the Company sell its prod
  ✅ What primary traits define the worldwide markets where the c
  ✅ In the risk factors section, how does the Company explain th
  ✅ Based on the Risk Factors portion, which elements of the Com
  ✅ What kind of statements are presented in the risk factors di
  ✅ Based on the excerpt, why is it challenging to audit managem
  ✅ Which particular procedures did the firm evaluate concerning
  ✅ Which elements might this evaluation be built upon?
  ✅ Which page of Apple’s FY 2024 Form 10‑K includes the Consoli
  ✅ Based on Note 1, whose financial records are incorporated in
  ✅ In the Index to the combined financial statements, which pag
  ✅ Which page of Apple’s FY 2024 Form 10‑K contains the Consoli
  ✅ In 2024, what was the aggregate amount of other current liab
  ✅ What amount of total equity did shareholders have on Septemb
  ✅ When are taxes allowed to be s

In [ ]:
# comparing Day 6's Ceiling Effect Scorecard against Today's real one
with open("day6_hit_rate_benchmark.json") as f:
    day6_results = json.load(f)

print(f"\n{'='*60}")
print("Day 6 (broken benchmark) vs Day 7 (hardened benchmark)")
print(f"{'='*60}")
print(f"{'Model':<40} {'Day 6':>10} {'Day 7':>10}")
print("-" * 62)
for d6, d7 in zip(day6_results, hardened_model_results):
    print(f"{d6['model_name']:<40} {d6['hit_rate']*100:>9.1f}% {d7['hit_rate']*100:>9.1f}%")

any_gap = any(r["hit_rate"] < 1.0 for r in hardened_model_results)
if any_gap:
    print("\n✅ Real separation finally appeared — Day 6's 100% ties were a ceiling effect, confirmed.")
    best = max(hardened_model_results, key=lambda x: x["hit_rate"])
    current = next(r for r in hardened_model_results if r["model_name"] == "BAAI/bge-m3")
    print(f"   bge-m3 (hardened): {current['hit_rate']*100:.1f}%  |  Best: {best['model_name']} at {best['hit_rate']*100:.1f}%")
else:
    print("\n📌 Still tied even on hard negatives + paraphrased questions — THIS result is now trustworthy,")
    print("   since the benchmark can clearly fail models when it needs to. Fine-tuning is genuinely not")
    print("   the highest-value next lever.")

with open("day7_hardened_benchmark.json", "w") as f:
    json.dump(hardened_model_results, f, indent=2)
print("\n📁 Saved to day7_hardened_benchmark.json")